# Lahore NDVI (Sentinel-2 L2A)
## Granularity: 10 m source imagery | Export: 100 m point GeoJSON


### 0. Initialize Earth Engine 


In [14]:
import calendar

import ee, geemap, geopandas as gpd, pandas as pd

ee.Authenticate()
ee.Initialize()

In [15]:
UC_SHP = "../../data/Lahore UCs/Lahore UC.shp"   # any Lahore boundary works; UC union is fine
YEAR = 2026
MONTH = 2
last_day = calendar.monthrange(YEAR, MONTH)[1]
START = f"{YEAR:04d}-{MONTH:02d}-01"
END = f"{YEAR:04d}-{MONTH:02d}-{last_day:02d}"
S2 = "COPERNICUS/S2_SR_HARMONIZED"
SAMPLE_SCALE = 100      # <-- 100 m keeps filesize sane. Avoid <50 m for whole Lahore.
SMOOTH_RADIUS_M = 0     # e.g., 30 or 50 to gently smooth NDVI (0 = no smoothing)
OUT_PREFIX = f"NDVI_Lahore_{calendar.month_abbr[MONTH]}{YEAR}"
print(f"Using date range: {START} to {END}")
print(f"Output prefix: {OUT_PREFIX}")

Using date range: 2026-02-01 to 2026-02-28
Output prefix: NDVI_Lahore_Feb2026


In [16]:
gdf = gpd.read_file(UC_SHP)
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
gdf["geometry"] = gdf["geometry"].buffer(0)
ucs_fc = geemap.gdf_to_ee(gdf)
region = ucs_fc.geometry()

# ---------- 2) Sentinel-2 NDVI (10 m), cloud-masked ----------
def mask_s2_sr(img):
    qa = img.select("QA60")
    cloud  = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus = qa.bitwiseAnd(1 << 11).eq(0)
    return img.updateMask(cloud.And(cirrus))

def add_ndvi(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return img.addBands(ndvi)

col = (ee.ImageCollection(S2)
       .filterBounds(region)
       .filterDate(START, END)
       .map(mask_s2_sr)
       .map(add_ndvi)
       .select("NDVI"))

ndvi_med = col.median().rename("NDVI")

# Optional gentle spatial smoothing (interpolation-look), in meters
if SMOOTH_RADIUS_M > 0:
    kernel = ee.Kernel.circle(radius=SMOOTH_RADIUS_M, units='meters', normalize=True)
    ndvi_med = ndvi_med.focal_mean(kernel=kernel, iterations=1)

# ---------- 3) Export POINT GRID (~SAMPLE_SCALE) ----------
# One feature per sample cell with lat/lon + NDVI value "val"
pts_fc = ee.Image.pixelLonLat().addBands(ndvi_med.rename("val")).sample(
    region=region,
    scale=SAMPLE_SCALE,
    geometries=True,
    seed=1
)

geemap.ee_export_vector(pts_fc, filename=f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m.geojson")
print(f"[OK] Saved → {OUT_PREFIX}_points_{SAMPLE_SCALE}m.geojson")

Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/NDVI/NDVI_Lahore_Feb2026_points_100m.geojson
[OK] Saved → NDVI_Lahore_Feb2026_points_100m.geojson
